# Retrieval ranking demo: TF-IDF vs. Dense vs. RRF

Synthetic CV (8 chunks emulating a parsed PDF) ranked against a backend job ad by:
- **TF-IDF** — hand-implemented vector-space model (`src/retrieval/tfidf_retriever.py`)
- **Dense** — sentence-transformers `all-MiniLM-L6-v2` (`src/retrieval/embedding_retriever.py`)
- **RRF** — Reciprocal Rank Fusion, `k_rrf=60` (`src/retrieval/fusion.py`)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.retrieval.tfidf_retriever import TfidfRetriever
from src.retrieval.embedding_retriever import DenseRetriever, load_minilm
from src.retrieval.fusion import reciprocal_rank_fusion

pd.set_option('display.max_colwidth', 80)

## Synthetic CV chunks (emulated PDF parse output)

Each chunk is a self-contained block — what `chunk_cv()` would emit for a real CV with role headers.

In [ ]:
CHUNKS = [
    "Jane Doe — Backend Software Engineer. Five years building REST and gRPC services in Python and Go, focused on data-intensive systems and reliable delivery.",
    "Acme Corp — Senior Backend Engineer    06/2023 – Present. Designed and shipped Python microservices on AWS (ECS, RDS Postgres, SQS). Built ETL pipelines processing 50M events/day. Owned on-call rotation and SLOs.",
    "Globex — Full-stack Developer    08/2020 – 05/2023. Wrote a TypeScript React dashboard, Node.js BFF, and a small Django admin. Mostly product work, some database modelling in Postgres.",
    "Kaggle competitions — Top 12% in two tabular competitions using gradient-boosted trees, feature engineering, and stacking. Wrote up methodology in Jupyter notebooks.",
    "Czech Technical University — BSc in Computer Science    09/2017 – 06/2020. Coursework in algorithms, databases, operating systems. Bachelor thesis on distributed consensus.",
    "Skills: Python, Go, PostgreSQL, Redis, Kafka, Docker, Kubernetes, AWS (ECS, RDS, SQS, S3), gRPC, REST, CI/CD with GitHub Actions, Terraform.",
    "Languages: English (C1), Czech (native), Spanish (B1). Volunteer mentor at Czechitas teaching backend fundamentals to career switchers since 2022.",
    "Open-source: maintainer of a small Python library for streaming data deduplication on top of Kafka. ~300 GitHub stars. Released semantic-versioned packages to PyPI.",
]

JOB_AD = (
    "We are hiring a Senior Backend Engineer to build scalable Python microservices on AWS. "
    "You will design REST and gRPC APIs, own data pipelines processing high event volumes, "
    "work with PostgreSQL, Redis, and Kafka, and ship code through CI/CD. "
    "Experience with Docker, Kubernetes, and observability is required."
)

## Fit retrievers on the corpus and query with the job ad

In [7]:
K = len(CHUNKS)

tfidf = TfidfRetriever()
tfidf.fit(CHUNKS)
tfidf_ranking = tfidf.query(JOB_AD, k=K)

model = load_minilm()
dense = DenseRetriever(model)
dense.fit(CHUNKS)
dense_ranking = dense.query(JOB_AD, k=K)

rrf_ranking = reciprocal_rank_fusion([tfidf_ranking, dense_ranking], k=K)

## Side-by-side comparison

Each row = one chunk. Columns show its rank (1 = best) under each method, plus a one-line preview.

In [ ]:
def ranks_from(ranking):
    return {cid: rank for rank, (cid, _) in enumerate(ranking, start=1)}

tfidf_ranks = ranks_from(tfidf_ranking)
dense_ranks = ranks_from(dense_ranking)
rrf_ranks = ranks_from(rrf_ranking)
tfidf_scores = dict(tfidf_ranking)
dense_scores = dict(dense_ranking)
rrf_scores = dict(rrf_ranking)

rows = []
for cid, text in enumerate(CHUNKS):
    rows.append({
        'chunk': cid,
        'preview': text[:70] + ('...' if len(text) > 70 else ''),
        'tfidf_rank': tfidf_ranks[cid],
        'tfidf_score': round(tfidf_scores[cid], 3),
        'dense_rank': dense_ranks[cid],
        'dense_score': round(dense_scores[cid], 3),
        'rrf_rank': rrf_ranks[cid],
        'rrf_score': round(rrf_scores[cid], 4),
    })

df = pd.DataFrame(rows).sort_values('rrf_rank').reset_index(drop=True)


def _color_rank(val: int) -> str:
    """Manual red->green gradient over ranks 1..N."""
    n = len(df)
    if n <= 1:
        return ''
    t = (val - 1) / (n - 1) # visuakizatibg with gradient
    r = int(round(120 + t * (235 - 120)))
    g = int(round(210 - t * (210 - 120)))
    b = 120
    return f'background-color: rgb({r},{g},{b}); color: black;'


df.style.applymap(_color_rank, subset=['tfidf_rank', 'dense_rank', 'rrf_rank'])

C:\Users\Lubomyr\AppData\Local\Temp\ipykernel_14528\3269864704.py:39: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  df.style.applymap(_color_rank, subset=['tfidf_rank', 'dense_rank', 'rrf_rank'])


,chunk,preview,tfidf_rank,tfidf_score,dense_rank,dense_score,rrf_rank,rrf_score
0,5,"Skills: Python, Go, PostgreSQL, Redis, Kafka, Docker, Kubernetes, AWS ...",1,0.433000,2,0.634000,1,0.032500
1,1,Acme Corp — Senior Backend Engineer 06/2023 – Present. Designed and...,2,0.302000,1,0.636000,2,0.032500
2,0,Jane Doe — Backend Software Engineer. Five years building REST and gRP...,3,0.113000,3,0.522000,3,0.031700
3,7,Open-source: maintainer of a small Python library for streaming data d...,4,0.035000,5,0.243000,4,0.031000
4,2,Globex — Full-stack Developer 08/2020 – 05/2023. Wrote a TypeScript...,5,0.031000,4,0.381000,5,0.031000
5,6,"Languages: English (C1), Czech (native), Spanish (B1). Volunteer mento...",6,0.008000,7,0.132000,6,0.030100
6,3,Kaggle competitions — Top 12% in two tabular competitions using gradie...,7,0.000000,6,0.132000,7,0.030100
7,4,Czech Technical University — BSc in Computer Science 09/2017 – 06/2...,8,0.000000,8,0.091000,8,0.029400


## Top-3 per method

TF-IDF rewards exact word/bigram matches, dense rewards paraphrases (chunk 7's "Kafka deduplication" matches the ad's "Kafka data pipelines" without sharing words). RRF combines both rankings, so chunks both methods agree on rise to the top.

In [5]:
def top3(ranking, label):
    print(f'{label}:')
    for rank, (cid, score) in enumerate(ranking[:3], start=1):
        preview = CHUNKS[cid][:65].replace('\n', ' ')
        print(f'  #{rank}  chunk {cid}  score={score:.4f}  | {preview}...')
    print()

top3(tfidf_ranking, 'TF-IDF')
top3(dense_ranking, 'Dense (MiniLM)')
top3(rrf_ranking, 'RRF (fused)')

TF-IDF:
  #1  chunk 5  score=0.4328  | Skills: Python, Go, PostgreSQL, Redis, Kafka, Docker, Kubernetes,...
  #2  chunk 1  score=0.3019  | Acme Corp — Senior Backend Engineer    06/2023 – Present. Designe...
  #3  chunk 0  score=0.1133  | Jane Doe — Backend Software Engineer. Five years building REST an...

Dense (MiniLM):
  #1  chunk 1  score=0.6356  | Acme Corp — Senior Backend Engineer    06/2023 – Present. Designe...
  #2  chunk 5  score=0.6337  | Skills: Python, Go, PostgreSQL, Redis, Kafka, Docker, Kubernetes,...
  #3  chunk 0  score=0.5224  | Jane Doe — Backend Software Engineer. Five years building REST an...

RRF (fused):
  #1  chunk 5  score=0.0325  | Skills: Python, Go, PostgreSQL, Redis, Kafka, Docker, Kubernetes,...
  #2  chunk 1  score=0.0325  | Acme Corp — Senior Backend Engineer    06/2023 – Present. Designe...
  #3  chunk 0  score=0.0317  | Jane Doe — Backend Software Engineer. Five years building REST an...

